# Uncertainty benchmark: train on Kaggle, evaluate anywhere

Trains the 7 models of the benchmark on CIFAR-10 and scores them against SVHN (far OOD),
CIFAR-100 (near OOD) and corrupted CIFAR-10:

| Method | Models trained |
|---|---|
| Softmax baseline | ResNet-18, seed 0 |
| Deep Ensemble | ResNet-18, seeds 0-4 (seed 0 is shared with the baseline) |
| MC Dropout | ResNet-18 with dropout in every block |
| Evidential Deep Learning | ResNet-18 with a Dirichlet output |

**Before running** (Notebook settings, right-hand panel):
1. **Accelerator:** GPU T4 x2 (or P100). Roughly an hour on T4 x2, about two hours on one GPU.
2. **Internet:** On. It is needed to clone the repo and download the datasets, and needs a phone-verified Kaggle account.

**Best way to run:** *Save Version* then *Save & Run All (Commit)*. It runs in the background,
and everything under `/kaggle/working` becomes downloadable output. Training skips models whose
checkpoint already exists, so a run that is cut short resumes where it stopped.

In [ ]:
import os
import subprocess
from pathlib import Path

import torch

# /kaggle/working: everything here becomes downloadable output. Set once, so
# running this cell again after the cd into the repo below doesn't move it.
if "HOME" not in globals():
    HOME = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()


def run(cmd: str):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A `!command` carries on after an error, and the last cell would then zip
    incomplete results that look complete.)"""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed with exit code {p.returncode}: {cmd}")


gpus = torch.cuda.device_count()
print("CUDA devices:", gpus, "| torch", torch.__version__)
if gpus == 0:
    raise RuntimeError("No GPU. Settings panel -> Accelerator -> GPU T4 x2, then run again.")
run("nvidia-smi -L")

## 1. Get the code

In [ ]:
repo = HOME / "uncertainty-rag"
if not repo.exists():
    run(f"git clone --depth 1 https://github.com/bhatsajid18/uncertainty-rag.git {repo}")
os.chdir(repo)
run("pip install -q pyyaml")

## 2. Smoke test (1 minute, no downloads)
Tiny fake datasets and networks. It checks the whole pipeline runs before the long job starts.

In [ ]:
run("python src/uncertainty/train.py --method all --fake --epochs 1")
run("python src/uncertainty/predict.py --fake --no-corruptions")
run("python src/uncertainty/evaluate.py --fake --no-figures")

## 3. Train all 7 models
30 epochs each, SGD with a cosine schedule and mixed precision. The work is spread over every visible GPU.

In [ ]:
run(f"python src/uncertainty/train.py --method all --gpus {gpus}")

## 4. Predict on every evaluation set
CIFAR-10 test, SVHN test, CIFAR-100 test, and 25 corrupted versions of CIFAR-10
(5 corruption types at 5 severities). MC Dropout makes 20 passes per input.

In [ ]:
run("python src/uncertainty/predict.py")

## 5. Tables and figures

In [ ]:
from IPython.display import Image, display

run("python src/uncertainty/evaluate.py")
for name in ["roc_svhn", "roc_cifar100", "corruption", "reliability", "risk_coverage"]:
    path = Path(f"results/uncertainty/figures/{name}.png")
    if path.exists():
        display(Image(str(path)))

## 6. Package the outputs for download
* `uncertainty_results.zip`: tables, figures, report and raw predictions (about 20 MB). This is what the web app's dashboard reads.
* `uncertainty_checkpoints.zip`: the 7 trained networks (about 300 MB). You only need these to re-run `predict.py` yourself.

In [ ]:
import shutil

run(f'zip -qr {HOME}/uncertainty_results.zip results/uncertainty -x "results/uncertainty/fake/*"')
run(f'zip -qr {HOME}/uncertainty_checkpoints.zip checkpoints -x "checkpoints/fake/*"')
run(f"ls -lh {HOME}/*.zip")
# The datasets (~700 MB) would otherwise be saved as notebook output too.
shutil.rmtree(repo / "data" / "torchvision", ignore_errors=True)

## Next, on your Mac
Download `uncertainty_results.zip` from the notebook's **Output** tab, then run this from the repo root:

```bash
unzip -o ~/Downloads/uncertainty_results.zip
python src/uncertainty/evaluate.py      # re-computes tables + figures from the predictions
```

Commit `results/uncertainty/` without `predictions/` (it's gitignored), and the dashboard shows the numbers.